# GISGPT — เริ่มต้นวิเคราะห์ภาพถ่ายดาวเทียมด้วย Python

Notebook แรกสำหรับทำความรู้จักกับข้อมูลภาพถ่ายดาวเทียม (Sentinel-2)

**สิ่งที่ต้องมี:** Python + rasterio, numpy, matplotlib (ติดตั้งแล้วในเครื่อง)

**ข้อมูลภาพ:** ถ้ายังไม่มีไฟล์ GeoTIFF จริง notebook จะสร้างข้อมูลตัวอย่าง (demo) ให้อัตโนมัติ เพื่อให้เรียนรู้ขั้นตอนได้เลย แล้วค่อยเปลี่ยนมาใช้ข้อมูลจริงภายหลัง

In [ ]:
import numpy as np
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path

plt.rcParams['figure.dpi'] = 110
for font in ['Tahoma', 'Segoe UI', 'Noto Sans Thai', 'Garuda']:
    try:
        font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except Exception:
        continue
print('libraries ready')

## 1. ข้อมูลภาพดาวเทียม (Sentinel-2) คืออะไร

- ดาวเทียม Sentinel-2 ของ ESA ถ่ายภาพทั้งโลกทุก ~5 วัน ความละเอียด **10 เมตร** ต่อพิกเซล
- ภาพมีหลาย **band** (ช่วงคลื่น) เช่น:
  - **B2 (Blue)**, **B3 (Green)**, **B4 (Red)** — สีที่ตามองเห็น (10 ม.)
  - **B8 (NIR, อินฟราเรดใกล้)** — พืชสะท้อน NIR สูงมาก (10 ม.) → ใช้วัดพื้นที่สีเขียว
  - B11/B12 (SWIR) — ใช้แยกอาคารกับดิน

**หัวใจของงานนี้:** พืชสะท้อนแสงสีเขียวเยอะ แต่มากกว่านั้นคือสะท้อน NIR เยอะมาก ในขณะที่น้ำดูดซับ NIR เกือบหมด ต่างกันชัดเจน → เอาไปคำนวณเป็นดัชนีได้

## 2. เตรียมภาพ

ถ้ามีไฟล์ GeoTIFF จริง (ดูวิธีดาวน์โหลดใน `docs/satellite-data.md`) ให้แก้ `IMAGE_PATH` ด้านล่าง
ถ้ายังไม่มี จะสร้างข้อมูลตัวอย่างให้อัตโนมัติ

In [ ]:
IMAGE_PATH = None  # เช่น Path('C:/data/sentinel2_roi.tif')

if IMAGE_PATH is not None and Path(IMAGE_PATH).exists():
    print(f'โหลดภาพจริง: {IMAGE_PATH}')
else:
    print('ไม่พบภาพจริง → สร้างข้อมูลตัวอย่าง (demo) เพื่อเรียนรู้ขั้นตอน')
    IMAGE_PATH = Path('demo_sentinel2.tif')
    rng = np.random.default_rng(42)
    size = 200
    y, x = np.mgrid[0:size, 0:size]
    green = (x - 60) ** 2 + (y - 130) ** 2 < 40 ** 2
    water = (x - 150) ** 2 + (y - 50) ** 2 < 25 ** 2
    built = (np.abs(x - 90) < 45) & (np.abs(y - 45) < 25)
    base = np.zeros((size, size))
    def scene(): return np.clip(base + rng.normal(0, 300, (size, size)), 0, None).astype('float32')
    b4 = scene() + built * 2600 + water * 150 + green * 400
    b3 = scene() + built * 2200 + water * 200 + green * 800
    b2 = scene() + built * 1800 + water * 1400 + green * 500
    b8 = scene() + built * 2800 + water * 100 + green * 3200
    with rasterio.open(
        IMAGE_PATH, 'w', driver='GTiff', height=size, width=size, count=4,
        dtype='float32', crs='EPSG:4326',
        transform=rasterio.transform.from_bounds(100.88, 13.07, 100.90, 13.09, size, size),
    ) as dst:
        dst.write(b2, 1)
        dst.write(b3, 2)
        dst.write(b4, 3)
        dst.write(b8, 4)
        dst.descriptions = ['B2 Blue', 'B3 Green', 'B4 Red', 'B8 NIR']
    print(f'สร้าง demo แล้ว: {IMAGE_PATH}')

## 3. อ่านภาพและสำรวจโครงสร้าง

ไฟล์ GeoTIFF เก็บข้อมูล 4 อย่างที่สำคัญ: ขนาดพิกเซล, จำนวน band, **CRS** (ระบบพิกัด) และ **bounds** (ขอบเขตพื้นที่)

In [ ]:
with rasterio.open(IMAGE_PATH) as src:
    print('ขนาด (height, width):', src.height, 'x', src.width)
    print('จำนวน band:', src.count)
    for i in range(1, src.count + 1):
        print(f'  band {i}:', src.descriptions[i - 1] if src.descriptions[i - 1] else '(ไม่มีชื่อ)')
    print('CRS:', src.crs)
    print('ขอบเขตพื้นที่ (degrees):', src.bounds)
    print('ความละเอียดต่อพิกเซล:', src.res)
    scale = 111.32  # km ต่อ 1 องศา (โดยประมาณ)
    w_km = src.width * src.res[0] * scale
    h_km = src.height * src.res[1] * scale
    print(f'ขนาดพื้นที่โดยประมาณ: {w_km:.2f} km x {h_km:.2f} km')

## 4. แสดงภาพสีจริง (True Color: RGB)

รวม band B4 (แดง), B3 (เขียว), B2 (น้ำเงิน) เข้าด้วยกันตามลำดับ RGB

In [ ]:
with rasterio.open(IMAGE_PATH) as src:
    red = src.read(3)
    green = src.read(2)
    blue = src.read(1)
    nir = src.read(4)

def normalize(band):
    lo, hi = np.percentile(band, 2), np.percentile(band, 98)
    return np.clip((band - lo) / (hi - lo), 0, 1)

rgb = np.stack([normalize(red), normalize(green), normalize(blue)], axis=-1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(rgb)
axes[0].set_title('True Color (RGB)')
axes[0].axis('off')
nir_viz = np.stack([normalize(nir), normalize(nir), normalize(nir)], axis=-1)
axes[1].imshow(nir_viz)
axes[1].set_title('NIR band (B8) — พืชสว่างมาก')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 5. ดัชนี NDVI — วัดพื้นที่สีเขียว

NDVI = (NIR − Red) / (NIR + Red)  → ค่าอยู่ระหว่าง −1 ถึง 1

- พืชเขียว: ~0.5–0.9
- ดิน/อาคาร: ~0.1–0.3
- น้ำ: < 0 (ติดลบ)

In [ ]:
ndvi = (nir - red) / (nir + red + 1e-6)
plt.figure(figsize=(5, 4.5))
im = plt.imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
plt.colorbar(im, label='NDVI')
plt.title('NDVI — เขียว = พืชสมบูรณ์')
plt.axis('off')
plt.show()
print(f'ค่า NDVI: ต่ำสุด {ndvi.min():.2f} | สูงสุด {ndvi.max():.2f}')

## 6. ดัชนี NDWI — หาแหล่งน้ำ

NDWI = (Green − NIR) / (Green + NIR)  → น้ำให้ค่าบวกสูง เนื่องจากน้ำดูดซับ NIR เกือบหมด

In [ ]:
ndwi = (green - nir) / (green + nir + 1e-6)
plt.figure(figsize=(5, 4.5))
im = plt.imshow(ndwi, cmap='BrBG', vmin=-0.5, vmax=0.5)
plt.colorbar(im, label='NDWI')
plt.title('NDWI — น้ำเงิน/เขียว = น้ำ')
plt.axis('off')
plt.show()
print(f'ค่า NDWI: ต่ำสุด {ndwi.min():.2f} | สูงสุด {ndwi.max():.2f}')

## 7. จำแนกชั้นการใช้ที่ดินแบบง่าย (threshold)

ใช้ NDVI + NDWI ตั้งค่าแบ่งชั้น (threshold) — จุดเริ่มต้นที่ง่ายที่สุดก่อนไปใช้ ML

- NDWI > 0.15 → น้ำ
- NDVI > 0.45 → พืชเขียว
- NDVI 0.25–0.45 → พืชบาง/ดิน
- ที่เหลือ → สิ่งปลูกสร้าง/ถนน

In [ ]:
classes = np.zeros(ndvi.shape, dtype='uint8')
classes[ndwi > 0.15] = 1
classes[ndvi > 0.45] = 2
classes[(ndvi > 0.25) & (ndvi <= 0.45)] = 3
classes[classes == 0] = 4
names = {0: 'unclassified', 1: 'water', 2: 'vegetation', 3: 'soil/sparse', 4: 'built-up'}
colors = plt.cm.Set3
plt.figure(figsize=(5, 4.5))
im = plt.imshow(classes, cmap=colors, vmin=0, vmax=4)
plt.colorbar(im, ticks=[0, 1, 2, 3, 4], label='class')
plt.title('Classification (threshold)')
plt.axis('off')
plt.show()

## 8. คำนวณพื้นที่ (km²) ของแต่ละชั้น

พิกเซลละ 10 ม. → 1 พิกเซล = 100 ตร.ม. = 0.0001 km² (กรณีข้อมูลจริงของ Sentinel-2)

In [ ]:
pixel_km2 = 0.0001  # 10 m x 10 m (ข้อมูลจริง) — demo ใช้ค่าเดียวกับข้อมูลจริงเพื่อความเข้าใจ
print(f'{"ชั้น":<22}{"พิกเซล":>10}{"พื้นที่ (km²)":>14}{"สัดส่วน":>10}')
print('-' * 56)
for c in sorted(names):
    count = int((classes == c).sum())
    print(f'{names[c]:<22}{count:>10}{count * pixel_km2:>14.3f}{count / classes.size * 100:>9.1f}%')
print('-' * 56)
print(f'{"รวม":<22}{classes.size:>10}{classes.size * pixel_km2:>14.3f}{"100.0%":>10}')

## 9. ขั้นต่อไป

1. ดาวน์โหลดภาพ Sentinel-2 จริง (ดู `docs/satellite-data.md`) แล้วเอามาใส่แทน demo
2. ลองเปลี่ยน threshold แล้วดูผลลัพธ์เปลี่ยนยังไง
3. เรียนรู้ Machine Learning (Random Forest) จำแนกชั้นโดยอัตโนมัติ
4. เชื่อมกับ prototype: ส่งพิกัด ROI จากแผนที่ → คำนวณผลลัพธ์จริง → แสดงผลกลับ